# Notebook: Tratamento da base CadÚnico

**id_rastreabilidade:** RTB_002  
**Versão:** v03  
**Data de criação:** 25/03/2026  
**Última atualização:** 08/05/2026  
**Responsável:** Ailton Vendramini  

---

## 🎯 Finalidade
Realizar a leitura, validação estrutural e tratamento técnico da base bruta do CadÚnico, preparando os dados para a camada limpa do projeto IVS-H.

---

## 📥 Fonte de Dados
**Fonte:** CadÚnico  
**Camada de entrada:** `01_bruto`  
**Camada de saída:** `02_limpo`  
**Período de referência da execução:** `2025_12`  
**Período de comparação:** —  
**Tipo de recorte temporal:** corte transversal  

**Arquivo de entrada:**  
`C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\cadunico\\01_bruto\\2025_12\\cadunico.csv`

**Arquivo de saída:**  
`C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\cadunico\\02_limpo\\2025_12\\cadunico_limpo.csv`

---

## 🧱 Base Conceitual
- `docs/metodologia_ivsh.md`
- `docs/regras_negocio.md`
- `docs/padrao_nomenclatura.md`

---

## ⚙️ Escopo técnico deste notebook

**Este notebook deve:**
1. Importações
2. Leitura da base bruta
3. Conferência de shape e colunas
4. Tipos de dados — diagnóstico e correção
5. Nulos — diagnóstico e tratamento
6. Datas — conversão para datetime
7. Criação de `idade`
8. Criação de `faixa_etaria`
9. Padronização textual mínima
10. Remoção de inconsistências gritantes
11. Exportação da base limpa

**Este notebook não deve:**
- calcular o IVS-H final
- misturar dados de outras fontes
- alterar diretamente arquivos da camada `01_bruto`

---

## 📊 Outputs esperados

| tipo_output | descrição | destino |
|---|---|---|
| exploratorio | validação inicial da base | não salvar |
| analitico | diagnósticos intermediários | `outputs/tabelas/cadunico/` |
| operacional | base limpa para próxima etapa | `dados/cadunico/02_limpo/2025_12/` |

---

## 🧠 Observações
- O arquivo bruto deve permanecer imutável.
- A data de referência para cálculo de idade é 31/12/2025 — data de corte do CadÚnico.
- Os nomes de colunas do CadÚnico possuem espaço no prefixo — removido na seção 2.
- Alterações estruturais relevantes devem ser registradas em nota técnica futura.

## 1. Importações

In [1]:
import pandas as pd       # leitura e manipulação de dados
import numpy as np        # operações numéricas e NaN
import os                 # criação de diretórios e verificação de caminhos
from datetime import date # data de referência para cálculo de idade

## 2. Leitura da base bruta

In [2]:
# constantes de referência
PERIODO_REFERENCIA = '2025_12'
DATA_REFERENCIA    = date(2025, 12, 31)

# caminho absoluto do arquivo bruto
caminho_entrada = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\01_bruto\2025_12\cadunico.csv'

# sep=';'            → CadÚnico usa ponto e vírgula como separador
# encoding='latin1'  → evita erro com caracteres acentuados
# low_memory=False   → lê tudo de uma vez e decide os tipos depois
df = pd.read_csv(caminho_entrada, sep=';', encoding='latin1', low_memory=False)

# os nomes de colunas do CadÚnico possuem espaço no prefixo (ex: ' p.dta_nasc_pessoa')
# .str.strip() remove espaços do início e fim de todos os nomes de uma vez
df.columns = df.columns.str.strip()

print(f'Base carregada: {df.shape[0]} linhas, {df.shape[1]} colunas')

Base carregada: 72424 linhas, 211 colunas


## 3. Conferência de shape e colunas

In [3]:
# shape esperado: 72424 linhas, 211 colunas
print(f'Linhas  : {df.shape[0]}')
print(f'Colunas : {df.shape[1]}')

# primeiras linhas para inspeção visual
df.head(3)

Linhas  : 72424
Colunas : 211


,d.cd_ibge,d.cod_familiar_fam,d.dat_cadastramento_fam,d.dat_atual_fam,d.cod_est_cadastral_fam,d.cod_forma_coleta_fam,d.dta_entrevista_fam,d.nom_localidade_fam,d.nom_tip_logradouro_fam,d.nom_titulo_logradouro_fam,...,p.ind_dinh_carregador_memb,p.ind_dinh_catador_memb,p.ind_dinh_servs_gerais_memb,p.ind_dinh_pede_memb,p.ind_dinh_vendas_memb,p.ind_dinh_outro_memb,p.ind_dinh_nao_resp_memb,p.ind_atend_nenhum_memb,p.ref_cad,p.ref_pbf
0,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
1,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
2,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0


## 4. Tipos de dados — diagnóstico e correção

In [4]:
# quantas colunas existem de cada tipo
print(df.dtypes.value_counts())

# colunas de renda devem ser numéricas
# errors='coerce' transforma o que não for número em NaN
colunas_renda = ['d.vlr_renda_media_fam', 'd.vlr_renda_total_fam']

for col in colunas_renda:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'{col} → {df[col].dtype}')

float64    134
object      46
int64       31
Name: count, dtype: int64
d.vlr_renda_media_fam → int64
d.vlr_renda_total_fam → int64


## 5. Nulos — diagnóstico e tratamento

In [5]:
# quantas colunas têm nulos e quantos são
nulos = df.isnull().sum()
nulos_presentes = nulos[nulos > 0].sort_values(ascending=False)
print(f'Colunas com nulos: {len(nulos_presentes)}')
print(nulos_presentes.head(20))

# no CadÚnico, o valor 999 indica ausência de resposta ou campo não aplicável
# substituímos por NaN para não contaminar cálculos
colunas_999 = [
    'p.cod_sabe_ler_escrever_memb',  # saber ler e escrever
    'p.cod_grau_instr_pes',           # grau de instrução
    'p.cod_sexo_pessoa',              # sexo
    'p.ind_frequenta_escola_memb'     # frequência escolar
]

for col in colunas_999:
    if col in df.columns:
        antes = (df[col] == 999).sum()
        df[col] = df[col].replace(999, np.nan)
        print(f'{col}: {antes} valores 999 → NaN')

Colunas com nulos: 171
d.cod_reserva_indigena_fam         72424
d.nom_reserva_indigena_fam         72424
d.cod_comunidade_quilombola_fam    72424
d.nom_comunidade_quilombola_fam    72424
d.nom_povo_indigena_fam            72418
d.cod_povo_indigena_fam            72418
d.cod_indigena_reside_fam          72418
p.nom_apelido_pessoa               72417
p.qtd_dormir_freq_dom_part_memb    72410
d.ind_risco_scl_vlco_drts          72399
p.ind_ajuda_outra_memb             72379
p.ind_ajuda_vizinho_memb           72372
p.qtd_freq_outro_memb              72342
p.ind_def_sindrome_down_memb       72299
p.qtd_dormir_freq_albergue_memb    72273
p.ind_ajuda_especializado_memb     72236
p.qtd_dormir_freq_rua_memb         72228
p.ind_ajuda_instituicao_memb       72193
p.ind_def_surdez_profunda_memb     72149
p.ind_def_cegueira_memb            72139
dtype: int64
p.cod_sabe_ler_escrever_memb: 0 valores 999 → NaN
p.cod_sexo_pessoa: 0 valores 999 → NaN
p.ind_frequenta_escola_memb: 0 valores 999 → NaN


## 6. Datas — conversão para datetime

In [6]:
# format='%d/%m/%Y' → formato do CadÚnico: dia/mês/ano
# errors='coerce'   → datas inválidas viram NaT (equivalente ao NaN para datas)
colunas_data = [
    'p.dta_nasc_pessoa',        # data de nascimento — usada para calcular idade
    'd.dat_cadastramento_fam',  # data de cadastramento da família
    'd.dat_atual_fam',          # data da última atualização
    'd.dta_entrevista_fam'      # data da entrevista
]

for col in colunas_data:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
        invalidas = df[col].isnull().sum()
        print(f'{col} → datetime | datas inválidas: {invalidas}')

p.dta_nasc_pessoa → datetime | datas inválidas: 0
d.dat_cadastramento_fam → datetime | datas inválidas: 0
d.dat_atual_fam → datetime | datas inválidas: 0
d.dta_entrevista_fam → datetime | datas inválidas: 14


## 7. Criação de `idade`

In [7]:
# idade calculada em relação à DATA_REFERENCIA (31/12/2025)
# isso garante reprodutibilidade — o resultado não muda conforme o tempo passa
# 365.25 → considera anos bissextos (mais preciso que 365)
# astype('Int64') → inteiro que aceita NaN (Int64 com I maiúsculo)
referencia = pd.Timestamp(DATA_REFERENCIA)

df['idade'] = (
    (referencia - df['p.dta_nasc_pessoa']).dt.days // 365.25
).astype('Int64')

print(f'Coluna idade criada')
print(f'Mínima : {df["idade"].min()}')
print(f'Máxima : {df["idade"].max()}')
print(f'Nulos  : {df["idade"].isnull().sum()}')

Coluna idade criada
Mínima : 0
Máxima : 100
Nulos  : 0


## 8. Criação de `faixa_etaria`

In [8]:
# faixas definidas conforme critérios do IVS-H e políticas socioassistenciais
def classificar_faixa(idade):
    if pd.isna(idade) or idade < 0:
        return 'invalida'
    elif idade < 6:
        return '00_05'
    elif idade < 15:
        return '06_14'
    elif idade < 18:
        return '15_17'
    elif idade < 30:
        return '18_29'
    elif idade < 60:
        return '30_59'
    else:
        return '60_mais'

df['faixa_etaria'] = df['idade'].apply(classificar_faixa)

print(df['faixa_etaria'].value_counts())

faixa_etaria
30_59      24672
06_14      13681
60_mais    11787
18_29      10831
00_05       7251
15_17       4202
Name: count, dtype: int64


## 9. Padronização textual mínima

In [9]:
# .str.strip() → remove espaços no início e no fim
# .str.upper() → converte para maiúsculas
# matching territorial completo ocorre no notebook de análise (03)
colunas_texto = [
    'd.nom_localidade_fam',  # nome do bairro/loteamento
    'd.nom_logradouro_fam'   # nome da rua
]

for col in colunas_texto:
    if col in df.columns:
        df[col] = df[col].str.strip().str.upper()
        print(f'{col} → padronizado')

d.nom_localidade_fam → padronizado
d.nom_logradouro_fam → padronizado


## 10. Remoção de inconsistências gritantes

In [10]:
# apenas inconsistências objetivas — casos ambíguos são documentados, não removidos

# idades inválidas: negativas ou acima de 120 anos
mask_idade_invalida = (df['idade'] < 0) | (df['idade'] > 120)
print(f'Idades inválidas (< 0 ou > 120): {mask_idade_invalida.sum()}')

# renda absurda: acima de 100 salários mínimos (R$ 150.400 em dez/2025)
LIMITE_RENDA = 150400
if 'd.vlr_renda_media_fam' in df.columns:
    mask_renda_absurda = df['d.vlr_renda_media_fam'] > LIMITE_RENDA
    print(f'Rendas absurdas (> R$ {LIMITE_RENDA}): {mask_renda_absurda.sum()}')

# nascimentos futuros em relação à data de referência
if 'p.dta_nasc_pessoa' in df.columns:
    mask_nasc_futuro = df['p.dta_nasc_pessoa'] > referencia
    print(f'Nascimentos futuros: {mask_nasc_futuro.sum()}')

Idades inválidas (< 0 ou > 120): 0
Rendas absurdas (> R$ 150400): 0
Nascimentos futuros: 0


## 11. Exportação da base limpa

In [11]:
# caminho absoluto de saída
caminho_saida = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\02_limpo\2025_12\cadunico_limpo.csv'

# garante que o diretório existe — não quebra se já existir
os.makedirs(os.path.dirname(caminho_saida), exist_ok=True)

# sep=';'          → mantém padrão do CadÚnico
# encoding='utf-8' → saída em UTF-8
# index=False      → não salva o índice do pandas como coluna
df.to_csv(caminho_saida, sep=';', encoding='utf-8', index=False)

print(f'Base limpa exportada com sucesso')
print(f'Linhas  : {df.shape[0]}')
print(f'Colunas : {df.shape[1]}')
print(f'Destino : {caminho_saida}')

Base limpa exportada com sucesso
Linhas  : 72424
Colunas : 213
Destino : C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\02_limpo\2025_12\cadunico_limpo.csv
